# X3 · Design patterns you already use

**Challenge track · self-study · about 60 minutes**

A *design pattern* is a named, reusable answer to a design problem that keeps coming back. You have been
using several of them for years without the names: every time you pass `scoring="roc_auc"`, build a
`Pipeline`, or call `fit_transform`. Knowing the names helps in two ways. You read library code faster,
because you recognise the shape ("ah, this is a registry"). And you can discuss designs with colleagues in
one word instead of ten sentences.

This notebook covers six patterns, each with: where it appears in scikit-learn, pandas or PyTorch, a small
example on the churn project, and when it is over-engineering.

**Learning goals.** By the end you can:

- recognise Strategy, Template Method, Composite, Factory/Registry, Adapter and fluent interfaces in library code
- implement a decorator-based registry and an adapter for the churn project
- explain a design choice to a colleague using the pattern's name and its trade-off
- judge when a pattern simplifies code and when a plain function or `if` statement is the better choice

**Before you start.** Notebooks 02 to 05 and 08. This notebook imports the reference package `churn`
(`src/churn`), so read its overview in notebook 11 first if you have not.

Exercises: **Core**, **Stretch**, **Challenge**. Check cells fail until you solve the exercise.

In [ ]:
import inspect
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split

from churn import ChurnModel, ConfusionMatrix, ExperimentConfig, RetentionCost, load_customers

DATA_PATH = Path("../data/raw/telco_churn.csv")
customers = load_customers(DATA_PATH)
X = customers.drop(columns=["Churn"])
y = customers["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = ChurnModel().fit(X_train, y_train)
proba_test = model.predict_proba(X_test)
print(model)

## 1. Strategy: swap the algorithm, keep the code around it

**Problem.** Picking the decision threshold is the same loop whatever we optimise: try thresholds, score
each, keep the best. Only the *scoring* differs (F1? money?). Copying the loop per metric duplicates code.

**Pattern.** Pass the varying part in as an object or function with a fixed interface. The surrounding code
calls it without knowing which one it got.

**In libraries.** `scoring=` in `GridSearchCV` and `cross_val_score` (a string or any callable), the
`criterion=` of trees, `solver=` in `LogisticRegression`, optimizers in PyTorch. In Python, functions are
objects, so a Strategy is often just a function. When it needs configuration (business numbers), it becomes
a callable object, like `RetentionCost` from notebook 03.

In [ ]:
THRESHOLDS = np.linspace(0.05, 0.95, 19)


def best_threshold(y_true, proba, loss, thresholds=THRESHOLDS):
    """Return the threshold with the lowest loss. `loss(y_true, y_pred)` is the strategy."""
    losses = [loss(y_true, (proba >= t).astype(int)) for t in thresholds]
    return float(thresholds[int(np.argmin(losses))])


def negative_f1(y_true, y_pred):
    return -f1_score(y_true, y_pred)


cheap_offers = RetentionCost(offer_cost=20)
expensive_offers = RetentionCost(offer_cost=150)

strategies = {"F1": negative_f1, "cheap offers": cheap_offers, "expensive offers": expensive_offers}
for name, loss in strategies.items():
    print(f"{name:>17}: best threshold {best_threshold(y_test, proba_test, loss):.2f}")

The loop did not change; the answer did. Cheap offers make it worth contacting almost every customer with
some risk (with these numbers an offer costs 20 and is expected to save 0.3 × 600 = 180 per churner reached,
so even a 10% churn probability pays off), expensive ones make the team pickier. (We tune on the test set here only to keep the demo short;
Exercise C does it properly with a validation split.)

`make_scorer` turns such a metric into the strategy interface scikit-learn expects for `scoring=`:

In [ ]:
def retention_cost_cheap(y_true, y_pred):
    # make_scorer's repr reads metric.__name__, which callable objects lack: wrap it in a function
    return cheap_offers(y_true, y_pred)


retention_scorer = make_scorer(retention_cost_cheap, greater_is_better=False)
numeric = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
scores = cross_val_score(
    LogisticRegression(max_iter=1000), X_train[numeric], y_train, cv=5, scoring=retention_scorer
)
# sklearn negates losses so that "higher is better" holds for every scorer
print("cost per fold:", (-scores).round(0))

`cross_val_score` knows nothing about retention offers. It calls whatever strategy it is given.

A small lesson about duck typing hides in that wrapper function: `make_scorer` accepts any callable, but its
`__repr__` (called inside `cross_val_score`) reads `metric.__name__`. Functions have that attribute, instances of
classes with `__call__` do not, so passing `cheap_offers` directly fails with an `AttributeError`. An
interface is everything the code actually uses, not only what the documentation mentions.

**Over-engineering alert.** If there is only ever one algorithm, a strategy parameter is noise. Add it when
the second variant appears.

## 2. Template Method: fix the skeleton, let subclasses fill in steps

**Problem.** Every transformer must remember to set a "fitted" flag and refuse `transform` before `fit`.
If each subclass writes that itself, one will forget.

**Pattern.** The base class implements the algorithm's skeleton as a concrete method and calls abstract
"hook" methods for the steps that vary.

**In libraries.** `TransformerMixin.fit_transform` (calls your `fit`, then your `transform`),
`torch.nn.Module.__call__` (runs hooks, then your `forward`), `unittest.TestCase.run` (setUp, test, tearDown).
In our package, `churn.features.Transformer`:

In [ ]:
from churn.features import Transformer

print(inspect.getsource(Transformer.fit))
print(inspect.getsource(Transformer.transform))

A subclass now only writes the interesting part, and the bookkeeping cannot be forgotten:

In [ ]:
class LogTransformer(Transformer):
    """log(1 + x) for skewed money columns such as TotalCharges."""

    def __init__(self, columns):
        super().__init__()
        self.columns = list(columns)

    def _fit(self, X):
        if (X[self.columns] < 0).any().any():
            raise ValueError("log1p needs non-negative values")

    def _transform(self, X):
        return X.assign(**{col: np.log1p(X[col]) for col in self.columns})


log_total = LogTransformer(["TotalCharges"])
try:
    log_total.transform(X_test)
except RuntimeError as exc:  # churn's NotFittedError is also a RuntimeError
    print(f"{type(exc).__name__}: {exc}")
print(log_total.fit(X_train).transform(X_test)["TotalCharges"].describe().round(2).to_dict())

**Over-engineering alert.** Template Method couples subclasses tightly to the base class. If the "steps"
are really independent pieces, composition (pass the pieces in) is usually more flexible.

## 3. Composite: a group that behaves like a single item

**Problem.** Preprocessing has several steps, but the model should treat "preprocessing" as *one* thing
with `fit` and `transform`.

**Pattern.** The container implements the same interface as its parts and forwards calls to them. Clients
cannot tell (and do not need to know) whether they hold one step or a tree of steps.

**In libraries.** `sklearn.pipeline.Pipeline` is itself an estimator, so pipelines nest and go into
`GridSearchCV`; `ColumnTransformer`, `FeatureUnion`; `torch.nn.Sequential`. In our package,
`FeaturePipeline` is a `Transformer` made of `Transformer`s:

In [ ]:
from churn.features import FeaturePipeline, OneHotEncoder, Standardizer

money = FeaturePipeline(
    [LogTransformer(["TotalCharges"]), Standardizer(["TotalCharges", "MonthlyCharges"])]
)
full = FeaturePipeline(
    [OneHotEncoder(["Contract"]), money],  # a pipeline inside a pipeline
    keep=["Contract", "tenure", "MonthlyCharges", "TotalCharges"],
)
print(full)
print(full.fit(X_train).transform(X_test).head(3).round(2))
print("full is a Transformer:", isinstance(full, Transformer))

**Over-engineering alert.** A composite pays off when groups really need to be treated like single items
(nesting, search over the whole thing). For a fixed two-step script, two lines of code are clearer.

## 4. Factory and Registry: create objects from names

**Problem.** A JSON config says `"model": "random_forest"`. Somewhere, a string must become an object.

**Pattern.** A *factory* is a function (or classmethod) that hides which class gets constructed and how.
A *registry* is a dictionary from names to factories, so new options can be added without editing the
factory: they register themselves.

**In libraries.** `ConfusionMatrixDisplay.from_predictions` and `pd.DataFrame.from_dict` are factory
classmethods; `sklearn.metrics.get_scorer("roc_auc")` looks a name up in a registry; Hugging Face
`AutoModel.from_pretrained(name)`. Our package has a simple factory with `if` statements:

In [ ]:
from churn.model import build_estimator

print(inspect.getsource(build_estimator))

That is fine for two models. With ten, or when other teams want to add models without touching your code,
a registry scales better. Python decorators make registration a one-liner, as you will build now.

### Exercise 4.1 · Core

Write a decorator `register(name)` that stores the decorated factory function in `MODEL_REGISTRY` under
`name` and returns the function unchanged. Then write `create_model(name, **params)` that looks the name up
and calls the factory, raising a `ValueError` that lists the known names if the name is unknown (a typo in a
config file should produce a helpful message, not a `KeyError`).

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

In [ ]:
MODEL_REGISTRY = {}


def register(name):
    raise NotImplementedError("Your code here")


def create_model(name, **params):
    raise NotImplementedError("Your code here")

In [ ]:
MODEL_REGISTRY.clear()  # so this cell can be re-run


@register("logistic_regression")
def make_logistic_regression(**params):
    return LogisticRegression(max_iter=1000, **params)


@register("random_forest")
def make_random_forest(**params):
    return RandomForestClassifier(random_state=42, **params)


@register("gradient_boosting")
def make_gradient_boosting(**params):
    return HistGradientBoostingClassifier(random_state=42, **params)


assert set(MODEL_REGISTRY) == {"logistic_regression", "random_forest", "gradient_boosting"}
assert make_random_forest.__name__ == "make_random_forest", "the decorator must return the function"
rf = create_model("random_forest", n_estimators=50)
assert isinstance(rf, RandomForestClassifier) and rf.n_estimators == 50
try:
    create_model("random_forrest")
    raise AssertionError("unknown names should raise ValueError")
except ValueError as exc:
    assert "random_forest" in str(exc), "the message should list the known names"
    print(f"ValueError: {exc}")

gb = ChurnModel(estimator=create_model("gradient_boosting")).fit(X_train, y_train)
print(f"gradient boosting ROC AUC: {roc_auc_score(y_test, gb.predict_proba(X_test)):.3f}")
print("All checks passed")

<details><summary>Hint</summary>

A decorator with an argument is a function that returns the actual decorator:
`def register(name): def decorator(factory): ...; return factory; return decorator`.
</details>

<details><summary>Solution</summary>

```python
MODEL_REGISTRY = {}


def register(name):
    def decorator(factory):
        if name in MODEL_REGISTRY:
            raise ValueError(f"A model called {name!r} is already registered")
        MODEL_REGISTRY[name] = factory
        return factory

    return decorator


def create_model(name, **params):
    try:
        factory = MODEL_REGISTRY[name]
    except KeyError:
        known = ", ".join(sorted(MODEL_REGISTRY))
        raise ValueError(f"Unknown model {name!r}. Known models: {known}") from None
    return factory(**params)
```

`from None` hides the internal `KeyError`: the caller made a typo, and the lookup detail would only distract.
Rejecting duplicate names protects against two modules silently overwriting each other's model.
</details>

**Over-engineering alert.** A registry adds indirection: "where is this class created?" becomes harder to
answer with go-to-definition. For a handful of options that only you maintain, the `if` version in
`build_estimator` is easier to read.

## 5. Adapter: make an object fit the interface you need

**Problem.** A function needs an object with interface A, you have one with interface B, and you cannot (or
should not) change either.

**Pattern.** Write a small wrapper that holds the original object and translates calls.

**In libraries.** `make_scorer` adapts a `metric(y_true, y_pred)` into a scorer `scorer(estimator, X, y)`
(section 1); `FunctionTransformer` adapts a plain function into a transformer; `torch.utils.data.Dataset`
subclasses adapt your files to what `DataLoader` expects; `pd.api.extensions` accessors.

### Exercise 5.1 · Stretch

The retention team's rule (flag month-to-month customers) only has `predict`. Our evaluation function below
needs `predict_proba` because it computes ROC AUC. Write `HardProbabilityAdapter`, which wraps any object
with `predict` and offers `predict_proba(X)` returning shape `(n, 2)` with columns `[1 - p, p]`, where `p` is
the 0/1 prediction. Forward `fit` too, returning the adapter.

Then answer: why is the rule's ROC AUC lower than the model's, even though it is a reasonable rule? What
does that tell you about comparing hard-label models with ROC AUC?

In [ ]:
class ContractRule:
    def fit(self, X, y):
        return self

    def predict(self, X):
        return (X["Contract"] == "Month-to-month").astype(int).to_numpy()


def roc_auc_of(model, X, y):
    """Works with any scikit-learn style classifier: predict_proba must return shape (n, 2)."""
    return round(roc_auc_score(y, model.predict_proba(X)[:, 1]), 3)

In [ ]:
class HardProbabilityAdapter:
    def __init__(self, model):
        raise NotImplementedError("Your code here")

    def fit(self, X, y):
        raise NotImplementedError("Your code here")

    def predict_proba(self, X):
        raise NotImplementedError("Your code here")

In [ ]:
adapted = HardProbabilityAdapter(ContractRule()).fit(X_train, y_train)
p = adapted.predict_proba(X_test.head(4))
assert p.shape == (4, 2) and np.allclose(p.sum(axis=1), 1)
assert set(np.unique(p)) <= {0.0, 1.0}
print("rule ROC AUC: ", roc_auc_of(adapted, X_test, y_test))
# ChurnModel.predict_proba returns only the churn column (a design choice), so it does not fit
# roc_auc_of's interface either; we call roc_auc_score directly
print("model ROC AUC:", round(roc_auc_score(y_test, model.predict_proba(X_test)), 3))
print("All checks passed")

<details><summary>Solution</summary>

```python
class HardProbabilityAdapter:
    """Expose predict_proba for a model that only predicts labels."""

    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        self.model.fit(X, y)
        return self

    def predict_proba(self, X):
        p = np.asarray(self.model.predict(X), dtype=float)
        return np.column_stack([1 - p, p])
```

ROC AUC measures how well a model *ranks* customers by risk across all thresholds. A hard 0/1 rule puts every
customer in one of two buckets, so it has exactly one operating point; the "curve" is two straight lines. That
caps its AUC even when its single operating point is decent. Compare hard rules with models at a fixed
threshold (precision, recall, cost), not by AUC. Notice also that `ChurnModel` itself did not fit `roc_auc_of`, because its
`predict_proba` returns one column: another small adapter (or a change to `ChurnModel`, since we own it).
</details>

**Over-engineering alert.** If you own both sides, change one of them instead of wrapping. Adapters are for
code you do not control (libraries, other teams).

## 6. Fluent interfaces and builders, briefly

`fit` returning `self` is what makes `ChurnModel().fit(X, y).predict(X)` possible: a *fluent* interface. pandas
method chaining (`df.assign(...).query(...).groupby(...)`) is the same idea. A *builder* goes one step further
and assembles a complex object through a series of calls. In Python, keyword arguments with defaults plus a
frozen dataclass usually make builders unnecessary. Variants of a config are one call away:

In [ ]:
base = ExperimentConfig()
variants = [replace(base, threshold=t) for t in (0.2, 0.3, 0.5)]
for config in variants:
    print(config.model, config.threshold)
try:
    replace(base, threshold=2.0)  # replace() runs __post_init__, so validation still applies
except ValueError as exc:
    print(f"{type(exc).__name__}: {exc}")

## 7. Putting it together

### Exercise 7.1 · Challenge

Build a small, honest threshold-selection procedure that combines Strategy and Registry:

1. A registry `OBJECTIVES` mapping names (`"f1"`, `"retention_cheap"`, `"retention_expensive"`) to loss
   callables `loss(y_true, y_pred) -> float` (lower is better). Reuse `negative_f1` and `RetentionCost`.
2. `select_threshold(model_factory, X, y, objective)` that splits `X, y` into train and **validation**
   (stratified, 25%, `random_state=42`), fits a fresh model from `model_factory()`, and returns the threshold
   from `best_threshold` on the validation set. The test set must not be touched.
3. For each objective: fit `ChurnModel` on all of `X_train` with the selected threshold, and report on
   `X_test` the number of flagged customers, recall, precision and the cost under `RetentionCost()` (default
   numbers). Collect the results in a DataFrame `threshold_report` indexed by objective.

Then discuss: which objective would you defend in front of the retention team's manager, and what
information would you need from the business to trust the cost numbers?

In [ ]:
OBJECTIVES = {}


def select_threshold(model_factory, X, y, objective):
    raise NotImplementedError("Your code here")


threshold_report = None  # a DataFrame indexed by objective name

In [ ]:
assert set(OBJECTIVES) == {"f1", "retention_cheap", "retention_expensive"}
assert isinstance(threshold_report, pd.DataFrame)
assert set(threshold_report.index) == set(OBJECTIVES)
assert {"threshold", "flagged", "recall", "precision", "cost"} <= set(threshold_report.columns)
cheap, expensive = (
    threshold_report.loc["retention_cheap"],
    threshold_report.loc["retention_expensive"],
)
assert cheap["threshold"] <= expensive["threshold"], (
    "cheaper offers should never lead to a stricter threshold"
)
print(threshold_report.round(3))
print("All checks passed")

<details><summary>Solution</summary>

```python
OBJECTIVES = {
    "f1": negative_f1,
    "retention_cheap": RetentionCost(offer_cost=20),
    "retention_expensive": RetentionCost(offer_cost=150),
}


def select_threshold(model_factory, X, y, objective):
    X_fit, X_val, y_fit, y_val = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    candidate = model_factory().fit(X_fit, y_fit)
    return best_threshold(y_val, candidate.predict_proba(X_val), OBJECTIVES[objective])


rows = {}
for name in OBJECTIVES:
    threshold = select_threshold(ChurnModel, X_train, y_train, name)
    final = ChurnModel(replace(ExperimentConfig(), threshold=threshold)).fit(X_train, y_train)
    y_pred = final.predict(X_test)
    cm = ConfusionMatrix.from_predictions(y_test, y_pred)
    rows[name] = {
        "threshold": threshold,
        "flagged": int(y_pred.sum()),
        "recall": cm.recall,
        "precision": cm.precision,
        "cost": RetentionCost()(y_test, y_pred),
    }
threshold_report = pd.DataFrame.from_dict(rows, orient="index")
```

Three patterns in a few lines: `OBJECTIVES` is a registry of strategies, `model_factory` is a factory passed
in (so the procedure works for any model, and a fresh model is built for validation), and
`DataFrame.from_dict` is a factory classmethod.

Look closely at the report. Every row is *evaluated* with the default cost (offers at 50), but the two
retention objectives were *tuned* assuming offers at 20 or 150. Both do worse than the F1 threshold under
the true cost: the cheap assumption contacts about a thousand customers, the expensive one flags almost
nobody (which is the same as doing nothing). An objective is only as good as the numbers behind it. The cost numbers are assumptions (offer cost, customer value,
acceptance rate); before defending an objective you would ask the business for those figures and ideally for
the results of a past campaign.
</details>

## Summary

- **Strategy**: pass the part that varies (a metric, a cost function) as a function or callable object.
- **Template Method**: a base class fixes the skeleton (`fit` sets the fitted flag) and calls hooks (`_fit`).
- **Composite**: a container with the same interface as its parts (`Pipeline`, `FeaturePipeline`).
- **Factory / Registry**: turn names from configs into objects; a registry lets others add options without editing your code.
- **Adapter**: a thin wrapper that translates one interface into another (`make_scorer`, `FunctionTransformer`).
- Every pattern adds indirection. Use it when the problem it solves is actually present, not in advance.

## Going further

- Notebook X2 shows how to make `ChurnModel`-like classes follow the scikit-learn contract.

## Sources and further reading

**Sources.** This notebook is based on:

- Erich Gamma, Richard Helm, Ralph Johnson and John Vlissides, *Design Patterns: Elements of Reusable Object-Oriented Software*, Addison-Wesley 1994: the original catalogue ("Gang of Four").
- [Brandon Rhodes, *Python Design Patterns*](https://python-patterns.guide/): which classic patterns Python makes unnecessary, and which still matter.
- [scikit-learn glossary](https://scikit-learn.org/stable/glossary.html) and [Buitinck et al. (2013), *API design for machine learning software: experiences from the scikit-learn project*](https://arxiv.org/abs/1309.0238): the patterns as scikit-learn uses them.

**For self-study.** Where to go next if you want to learn more:

- [Refactoring Guru: Design Patterns](https://refactoring.guru/design-patterns) (website, intermediate): illustrated catalogue with Python examples.
- [Peter Norvig, *Design Patterns in Dynamic Languages*](https://norvig.com/design-patterns/) (slides, advanced): why 16 of the 23 classic patterns become simpler or invisible in languages like Python.
- Luciano Ramalho, *Fluent Python*, 2nd edition, O'Reilly 2022 ([book site](https://www.fluentpython.com/)), chapter 10 "Design Patterns with First-Class Functions" (book, advanced).